In [ ]:
%load_ext autoreload 
%autoreload 2
%matplotlib inline
import sys
import cProfile #for checking the nr of calls and execution time
import pstats
from pstats import SortKey
from scipy.stats import gaussian_kde
import json
import os
import re
import numpy as np
import multiple_planets_gas_acc as code_gas
from functions_pebble_accretion import *
from functions import *
import functions_plotting as plot
import sim_loader as sim_load
import functions_plotting as plot
import matplotlib.pyplot as plt
import matplotlib as mpl
import astropy.units as u
import pandas as pd
from matplotlib.ticker import ScalarFormatter, LogFormatter, LogLocator, MultipleLocator, AutoMinorLocator
from matplotlib import cm, ticker
from matplotlib import colors
import matplotlib.gridspec as gridspec
import matplotlib.patches as patch
from matplotlib.offsetbox import AnchoredText
from matplotlib.patches import Patch
import matplotlib.lines as mlines 
from matplotlib.animation import FuncAnimation
from matplotlib import pyplot as plt, ticker as mticker
import sim_loader as sim_load
import h5py
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
    "axes.labelsize": 15,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.direction": "in",
    "ytick.direction": "in",
    # major ticks
    "xtick.major.size": 12,
    "ytick.major.size": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    # minor ticks
    "xtick.minor.size": 8,
    "ytick.minor.size": 8,
    "xtick.minor.visible": True,   # minor ticks are hidden by default — need this to actually see them
    "ytick.minor.visible": True,

})

In [ ]:
# ============================================================
# FILE SORTING
# ============================================================

def extract_initial_time(filename):
    with h5py.File(filename, 'r') as f:
        time = f['simulation']['time'][()]
        if np.ndim(time) > 0:
            return float(time[0])
        return float(time)

In [ ]:
# ============================================================
# LOAD SIMULATIONS 
# ============================================================
folder_path_multi = ["sims/opacities/1Msun/Mdot_scatter/double_planet/100_pairs/1_2_scales_closer"]

simulations_multi = []
sim_parameters_multi = []
parameters_multi = []
simulations_single = []
sim_parameters_single = []
parameters_single = []

for folder in folder_path_multi:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations_multi.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters_multi.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters_multi.append([sim_load.load_params(f) for f in h5_files])

In [ ]:
alpha_transp = 0.2
print("Number of simulations loaded: ", len(simulations_multi[0]))
c = plt.cm.inferno(np.linspace(0, 1, len(simulations_multi[0])))
print(len(c))
fig, axs = plt.subplots(1, 1, figsize=(6, 6))

for i, folder in enumerate(folder_path_multi):
    for j, (sim, params, sim_params) in enumerate(zip(simulations_multi[i], parameters_multi[i], sim_parameters_multi[i])):
        col = c[j]
        for p in range(sim_params.nr_planets):
            idx = plot.idxs(sim.time.value, sim.mass[p].value, sim.position[p].value, sim.filter_fraction[p],
                             sim.dR_dt[p].value, sim.dM_dt[p].value, params, True)
            axs.scatter(sim.position[p, -1], sim.mass[p, -1], color=col)

## aesthetics of the plot
a_p0 = np.geomspace(0.01, 100, 100)
axs.loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
axs.loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
axs.fill_between(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params),  color='slateblue', alpha=0.1)   

axs.set_xlabel('r [AU]', fontsize=25, labelpad=20)
axs.set_xlim(0.01, 1e2)
axs.set_ylim(1e-3, 1e4)
axs.axhline(13*u.M_jup.to(u.M_earth), color = 'grey', linestyle = '--', alpha = 0.5)
axs.set_ylabel('M [$M_{\oplus}$]', fontsize=25, labelpad=20)
axs.set_title("Doubles, $\dot{M}_{\star} = 1/2 \dot{M}_{\star, 0}$, 1/2 migration", fontsize=15, pad=20)


In [ ]:
alpha_transp = 0.2
c = ['teal','orange']

fig, axs = plt.subplots(1, 1, figsize=(6, 6))
for sim, params, sim_params in zip(simulations_multi[0], parameters_multi[0], sim_parameters_multi[0]):
    # axs.axvline(r_magnetic_cavity(M_dot_star(sim_params.t_in, params), params), linestyle = '-.', color = 'grey', alpha = 0.1)
    # axs.axvline(r_magnetic_cavity(M_dot_star(sim_params.t_fin, params), params), linestyle = '-.', color = 'grey', alpha = 0.1)
    axs.loglog(sim.position[:, -1], sim.mass[:, -1], color='grey', alpha=0.1, zorder = 0)
    for p in range(sim_params.nr_planets):
        axs.scatter(sim.position[p, -1], sim.mass[p, -1], color=c[p])

## aesthetics of the plot
a_p0 = np.geomspace(0.01, 100, 100)
axs.loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
axs.loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
axs.fill_between(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params),  color='slateblue', alpha=0.1)   

axs.set_ylim(1e-3, 1e4)
axs.axhline(13*u.M_jup.to(u.M_earth), color = 'grey', linestyle = '--', alpha = 0.5)
axs.set_ylabel('$M_{\\rm p}$ [$M_{\oplus}$]', fontsize=25, labelpad=20)
axs.set_xlabel('$a_{\\rm p}$ [AU]', fontsize=25, labelpad=20)
axs.set_title("Two planets", fontsize=15, pad=20)
plt.savefig("figures/opacities/double_planet_sims_M_a2_1_2_scales_closer.png", bbox_inches='tight')


In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6, 6))
c = ['teal','orange']

for sim, params, sim_params in zip(simulations_multi[0], parameters_multi[0], sim_parameters_multi[0]):
    axs.loglog(sim.position[:, -1], sim.mass[:, -1]/sim.position[:, -1]**2, color='grey', alpha=0.1, zorder = 0)
    for p in range(sim_params.nr_planets):
        axs.scatter(sim.position[p, -1], sim.mass[p, -1]/sim.position[p, -1]**2,  color=c[p])
axs.set_xlabel('$a_{\\rm p}$ [AU]', fontsize=25, labelpad=20)
axs.set_ylabel('$M_{\\rm p}/a_{\\rm p}$ [$M_{\oplus}$/AU]', fontsize=25, labelpad=20)
axs.set_xscale('log')
axs.set_yscale('log')
axs.set_ylim(1e-3, 1e4)
axs.set_title("Two planets sims", fontsize=15, pad=20)
plot.all_y_ticks(axs, 1000)
plt.savefig("figures/opacities/double_planet_sims_M_a2_1_2_scales_closer.png", bbox_inches='tight')